# Chapter 5: Celadon City -- Regression, Weighting & Doubly Robust

---

Welcome to **Celadon City**, the sprawling commercial heart of Kanto.  
Home to the massive Celadon Department Store, the Game Corner, and Erika's Grass-type Gym,  
this city is where trainers *spend* -- on TMs, Potions, Rare Candies, and everything in between.

But does spending at the Department Store actually *cause* trainers to earn more badges?  
Or are wealthier, more experienced trainers simply shopping more *and* winning more?

In this chapter we will use **regression**, **inverse probability weighting (IPW)**, and  
**doubly robust estimation** to tease apart causal effects from confounded associations.  
Along the way, we will learn about:

- Omitted Variable Bias (OVB)  
- Bad controls (post-treatment variables)  
- Propensity score weighting and diagnostics  
- The doubly robust property  
- Sensitivity analysis for unobserved confounding  

---
## 5.1 Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Ensure kanto_utils is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from kanto_utils import (
    load_trainers, apply_kanto_theme,
    propensity_score, ipw_estimate, doubly_robust,
    oak_says, blue_says, blues_mistake, badge_earned,
)

apply_kanto_theme()
np.random.seed(151)  # Kanto Pokedex count

# Load the Kanto trainers dataset
trainers = load_trainers()
print(f"Loaded {len(trainers)} trainers with {trainers.shape[1]} variables.")
trainers.head()

In [ ]:
oak_says(
    "Welcome to Celadon City! The Department Store is the largest shop in Kanto -- "
    "trainers can buy TMs, vitamins, evolution stones, and more. "
    "Our research question: <b>Does using the Exp. Share cause trainers to earn more badges?</b> "
    "We will need to control for confounders like wealth, experience, and strategy "
    "to isolate the causal effect."
)

---
## 5.2 Regression for Causal Inference

In [ ]:
# ---- Naive regression: badges ~ dept_store_spending ----
naive = smf.ols('badges ~ dept_store_spending', data=trainers).fit()
print("=== NAIVE REGRESSION (no controls) ===")
print(naive.summary().tables[1])
print(f"\nNaive estimate: 1 more unit of spending -> {naive.params['dept_store_spending']:.5f} more badges")
print(f"R-squared: {naive.rsquared:.4f}")

In [ ]:
# ---- Add confounders: wealth and trainer_experience ----
controlled = smf.ols('badges ~ dept_store_spending + wealth + trainer_experience', data=trainers).fit()
print("=== CONTROLLED REGRESSION (+ wealth + experience) ===")
print(controlled.summary().tables[1])

ovb = naive.params['dept_store_spending'] - controlled.params['dept_store_spending']
print(f"\nNaive coefficient:      {naive.params['dept_store_spending']:.5f}")
print(f"Controlled coefficient: {controlled.params['dept_store_spending']:.5f}")
print(f"Omitted Variable Bias:  {ovb:.5f}")

In [ ]:
oak_says(
    "The coefficient on <code>dept_store_spending</code> changed when we added controls! "
    "This is <b>Omitted Variable Bias</b> in action. The naive estimate was picking up "
    "the effect of wealth and experience, which are correlated with both spending and badges. "
    "The OVB formula: bias = (short coefficient) - (long coefficient). "
    "A positive bias means confounders were inflating our estimate."
)

---
## 5.3 Sequential Covariate Addition

In [ ]:
# Add covariates one at a time and track how the treatment coefficient changes
covariate_sequence = ['wealth', 'trainer_experience', 'strategy_score', 'play_hours', 'cave_training']

results = []
# Start with the naive model
formula = 'badges ~ dept_store_spending'
mod = smf.ols(formula, data=trainers).fit()
results.append({
    'controls': 'None',
    'coef': mod.params['dept_store_spending'],
    'se': mod.bse['dept_store_spending'],
    'r2': mod.rsquared
})

controls_so_far = []
for cov in covariate_sequence:
    controls_so_far.append(cov)
    formula = 'badges ~ dept_store_spending + ' + ' + '.join(controls_so_far)
    mod = smf.ols(formula, data=trainers).fit()
    results.append({
        'controls': ' + '.join(controls_so_far),
        'coef': mod.params['dept_store_spending'],
        'se': mod.bse['dept_store_spending'],
        'r2': mod.rsquared
    })

results_df = pd.DataFrame(results)
display(results_df[['controls', 'coef', 'se', 'r2']].round(5))

In [ ]:
# Visualise how the coefficient evolves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: coefficient path
x_pos = range(len(results_df))
ax1.errorbar(
    x_pos, results_df['coef'],
    yerr=1.96 * results_df['se'],
    fmt='o-', color='#EE1515', capsize=5, linewidth=2, markersize=8
)
ax1.set_xticks(list(x_pos))
ax1.set_xticklabels(['None'] + covariate_sequence, rotation=35, ha='right')
ax1.set_ylabel('Coefficient on dept_store_spending')
ax1.set_xlabel('Covariates added')
ax1.set_title('Coefficient Stability as Controls are Added')
ax1.axhline(0, color='gray', ls=':', lw=1)

# Right: OVB at each step
ovb_values = [results_df['coef'].iloc[0] - results_df['coef'].iloc[i] for i in range(len(results_df))]
ax2.bar(x_pos, ovb_values, color='#3B4CCA', edgecolor='white')
ax2.set_xticks(list(x_pos))
ax2.set_xticklabels(['None'] + covariate_sequence, rotation=35, ha='right')
ax2.set_ylabel('Cumulative OVB (naive - current)')
ax2.set_xlabel('Covariates added')
ax2.set_title('Omitted Variable Bias by Stage')

fig.tight_layout()
plt.show()

In [ ]:
oak_says(
    "As we sequentially add covariates, watch the coefficient stabilise. "
    "If the coefficient changes dramatically when you add a variable, "
    "that variable was a <b>confounding factor</b>. If it barely moves, "
    "the variable was not confounding the treatment-outcome relationship. "
    "Stability of the coefficient across specifications gives us "
    "<b>informal evidence</b> that we may have accounted for the key confounders."
)

---
## 5.4 The Bad Controls Trap

In [ ]:
# team_level_avg is a POST-TREATMENT variable -- it is affected by exp_share_used.
# Controlling for it introduces post-treatment bias.

# Good model: control for pre-treatment confounders only
good_model = smf.ols(
    'badges ~ exp_share_used + wealth + trainer_experience + strategy_score',
    data=trainers
).fit()

# Bad model: add post-treatment variable team_level_avg
bad_model = smf.ols(
    'badges ~ exp_share_used + wealth + trainer_experience + strategy_score + team_level_avg',
    data=trainers
).fit()

print("=== GOOD MODEL (pre-treatment controls only) ===")
print(f"  exp_share_used coeff: {good_model.params['exp_share_used']:.4f} "
      f"(SE: {good_model.bse['exp_share_used']:.4f}, p={good_model.pvalues['exp_share_used']:.4f})")
print(f"  R-squared: {good_model.rsquared:.4f}")

print("\n=== BAD MODEL (includes post-treatment team_level_avg) ===")
print(f"  exp_share_used coeff: {bad_model.params['exp_share_used']:.4f} "
      f"(SE: {bad_model.bse['exp_share_used']:.4f}, p={bad_model.pvalues['exp_share_used']:.4f})")
print(f"  R-squared: {bad_model.rsquared:.4f}")

print(f"\nBias from bad control: {bad_model.params['exp_share_used'] - good_model.params['exp_share_used']:.4f}")

In [ ]:
blues_mistake(
    "I controlled for team_level_avg and the Exp. Share effect disappeared! "
    "So Exp. Share doesn't really help -- it's all about having high-level Pokemon.",
    "team_level_avg is a <b>post-treatment variable</b> -- it is itself caused by "
    "Exp. Share usage. Controlling for it blocks part of the causal pathway "
    "(Exp. Share -> higher team levels -> more badges). The DAG shows: "
    "exp_share_used -> team_level_avg -> badges. "
    "We should NEVER control for a mediator if we want the total causal effect."
)

In [ ]:
# Draw a simple DAG showing why team_level_avg is a bad control
fig, ax = plt.subplots(figsize=(9, 4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('DAG: Why team_level_avg is a Bad Control', fontsize=14, fontweight='bold')

# Nodes
nodes = {
    'Exp. Share': (1.5, 2.5),
    'Team Level Avg\n(mediator)': (5, 2.5),
    'Badges': (8.5, 2.5),
    'Wealth /\nExperience': (3.25, 4.5),
}
for label, (x, y) in nodes.items():
    color = '#EE1515' if 'mediator' in label else '#3B4CCA'
    ax.annotate(
        label, (x, y), fontsize=11, ha='center', va='center',
        fontweight='bold', color='white',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.85)
    )

# Arrows
arrow_kw = dict(arrowstyle='->', color='#333', lw=2)
ax.annotate('', xy=(4, 2.5), xytext=(2.8, 2.5), arrowprops=arrow_kw)
ax.annotate('', xy=(7.3, 2.5), xytext=(6.2, 2.5), arrowprops=arrow_kw)
ax.annotate('', xy=(2.0, 3.2), xytext=(2.8, 4.0), arrowprops=arrow_kw)
ax.annotate('', xy=(7.8, 3.2), xytext=(4.0, 4.3), arrowprops=arrow_kw)

# X through the bad path
ax.text(5, 1.2, 'Controlling for the mediator BLOCKS the causal path!',
        fontsize=10, ha='center', color='#EE1515', fontstyle='italic')

plt.tight_layout()
plt.show()

---
## 5.5 Inverse Probability Weighting (IPW)

In [ ]:
# Treatment: exp_share_used (binary)
# Outcome: badges
# Confounders for propensity score
confounders = ['wealth', 'trainer_experience', 'strategy_score', 'play_hours']

X_ps = trainers[confounders].values
treatment = trainers['exp_share_used'].values
outcome = trainers['badges'].values

# Estimate propensity scores
ps = propensity_score(X_ps, treatment)

print(f"Propensity score summary:")
print(f"  Min:    {ps.min():.4f}")
print(f"  Median: {np.median(ps):.4f}")
print(f"  Max:    {ps.max():.4f}")
print(f"  Mean:   {ps.mean():.4f}")

In [ ]:
# Horvitz-Thompson vs Hajek estimators
ht_result = ipw_estimate(outcome, treatment, ps, estimator='ht')
hajek_result = ipw_estimate(outcome, treatment, ps, estimator='hajek')

print("=== Horvitz-Thompson (unnormalised) ===")
print(f"  ATE estimate: {ht_result['estimate']:.4f}")
print(f"  95% CI: [{ht_result['ci_lower']:.4f}, {ht_result['ci_upper']:.4f}]")

print("\n=== Hajek (normalised) ===")
print(f"  ATE estimate: {hajek_result['estimate']:.4f}")
print(f"  95% CI: [{hajek_result['ci_lower']:.4f}, {hajek_result['ci_upper']:.4f}]")

In [ ]:
# Propensity score distribution and weight diagnostics
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (a) PS distribution by treatment group
ax = axes[0]
ax.hist(ps[treatment == 1], bins=30, alpha=0.6, color='#EE1515', label='Treated (Exp Share=1)', density=True)
ax.hist(ps[treatment == 0], bins=30, alpha=0.6, color='#3B4CCA', label='Control (Exp Share=0)', density=True)
ax.set_xlabel('Propensity Score')
ax.set_ylabel('Density')
ax.set_title('PS Distribution by Group')
ax.legend(fontsize=9)

# (b) IPW weights distribution
ax = axes[1]
weights = np.where(treatment == 1, 1.0 / ps, 1.0 / (1.0 - ps))
ax.hist(weights, bins=50, color='#FFD733', edgecolor='white', alpha=0.8)
ax.axvline(np.percentile(weights, 99), color='#EE1515', ls='--', lw=2, label=f'99th pctl = {np.percentile(weights, 99):.1f}')
ax.set_xlabel('IPW Weight')
ax.set_ylabel('Count')
ax.set_title('IPW Weight Distribution')
ax.legend(fontsize=9)

# (c) Flag extreme weights
ax = axes[2]
extreme_threshold = np.percentile(weights, 95)
is_extreme = weights > extreme_threshold
ax.scatter(ps[~is_extreme], weights[~is_extreme], s=15, alpha=0.4, color='#3B4CCA', label='Normal')
ax.scatter(ps[is_extreme], weights[is_extreme], s=30, alpha=0.8, color='#EE1515', label='Extreme (>95th pctl)')
ax.set_xlabel('Propensity Score')
ax.set_ylabel('IPW Weight')
ax.set_title('Weights vs Propensity Score')
ax.legend(fontsize=9)

fig.tight_layout()
plt.show()

print(f"Number of extreme weights (>95th pctl = {extreme_threshold:.2f}): {is_extreme.sum()}")
print(f"Max weight: {weights.max():.2f}")

---
## 5.6 Weight Diagnostics: Trimming & Effective Sample Size

In [ ]:
# Compare trimmed vs untrimmed estimates at various thresholds
trim_thresholds = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20]
trim_results = []

for trim in trim_thresholds:
    ps_trimmed = np.clip(ps, trim, 1 - trim)
    res = ipw_estimate(outcome, treatment, ps_trimmed, estimator='hajek')
    
    # Effective sample size
    w_t = treatment / ps_trimmed
    w_c = (1 - treatment) / (1 - ps_trimmed)
    ess_t = (np.sum(w_t))**2 / np.sum(w_t**2) if np.sum(w_t) > 0 else 0
    ess_c = (np.sum(w_c))**2 / np.sum(w_c**2) if np.sum(w_c) > 0 else 0
    
    trim_results.append({
        'trim_level': trim,
        'estimate': res['estimate'],
        'se': res['se'],
        'ci_lower': res['ci_lower'],
        'ci_upper': res['ci_upper'],
        'ess_treated': ess_t,
        'ess_control': ess_c,
    })

trim_df = pd.DataFrame(trim_results)
display(trim_df.round(4))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: estimate vs trimming threshold
ax1.errorbar(
    trim_df['trim_level'], trim_df['estimate'],
    yerr=1.96 * trim_df['se'],
    fmt='o-', color='#EE1515', capsize=5, linewidth=2, markersize=8
)
ax1.axhline(hajek_result['estimate'], color='#3B4CCA', ls='--', lw=1.5, label=f'Untrimmed = {hajek_result["estimate"]:.3f}')
ax1.set_xlabel('Trimming threshold')
ax1.set_ylabel('IPW (Hajek) Estimate')
ax1.set_title('IPW Estimate vs Trimming Level')
ax1.legend(fontsize=9)

# Right: effective sample size
ax2.plot(trim_df['trim_level'], trim_df['ess_treated'], 'o-', color='#EE1515', lw=2, label='Treated ESS')
ax2.plot(trim_df['trim_level'], trim_df['ess_control'], 's-', color='#3B4CCA', lw=2, label='Control ESS')
ax2.set_xlabel('Trimming threshold')
ax2.set_ylabel('Effective Sample Size')
ax2.set_title('Effective Sample Size vs Trimming')
ax2.legend(fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
oak_says(
    "<b>Weight trimming</b> clips extreme propensity scores away from 0 and 1, "
    "preventing any single observation from dominating the estimate. "
    "The trade-off: more trimming reduces variance (and extreme weights) but "
    "introduces some bias by changing the target population. "
    "The <b>effective sample size (ESS)</b> tells us how many independent "
    "observations our weighted sample is worth -- lower ESS means more variable estimates."
)

---
## 5.7 Doubly Robust Estimation

In [ ]:
# --- Doubly Robust (AIPW) with correctly specified models ---
X_dr = trainers[confounders].values
dr_correct = doubly_robust(outcome, treatment, X_dr)
print("=== Doubly Robust (both models correct) ===")
print(f"  ATE: {dr_correct['estimate']:.4f}")
print(f"  SE:  {dr_correct['se']:.4f}")
print(f"  95% CI: [{dr_correct['ci_lower']:.4f}, {dr_correct['ci_upper']:.4f}]")

In [ ]:
# --- Misspecification experiment ---
# (a) WRONG propensity score model, correct outcome model
# Use a deliberately bad PS: only use a single irrelevant predictor
X_bad_ps = trainers[['fishing_attempts']].values
ps_wrong = propensity_score(X_bad_ps, treatment)
dr_wrong_ps = doubly_robust(outcome, treatment, X_dr, propensity_scores=ps_wrong)

# (b) Correct PS, WRONG outcome model
# Use a single irrelevant covariate in the outcome model
X_bad_outcome = trainers[['fishing_attempts']].values
ps_correct = propensity_score(X_dr, treatment)
dr_wrong_outcome = doubly_robust(outcome, treatment, X_bad_outcome, propensity_scores=ps_correct)

# (c) BOTH models wrong
dr_both_wrong = doubly_robust(outcome, treatment, X_bad_outcome, propensity_scores=ps_wrong)

# Summary table
dr_results = pd.DataFrame([
    {'Scenario': 'Both correct', 'ATE': dr_correct['estimate'], 'SE': dr_correct['se'],
     'CI_lower': dr_correct['ci_lower'], 'CI_upper': dr_correct['ci_upper']},
    {'Scenario': 'Wrong PS, correct outcome', 'ATE': dr_wrong_ps['estimate'], 'SE': dr_wrong_ps['se'],
     'CI_lower': dr_wrong_ps['ci_lower'], 'CI_upper': dr_wrong_ps['ci_upper']},
    {'Scenario': 'Correct PS, wrong outcome', 'ATE': dr_wrong_outcome['estimate'], 'SE': dr_wrong_outcome['se'],
     'CI_lower': dr_wrong_outcome['ci_lower'], 'CI_upper': dr_wrong_outcome['ci_upper']},
    {'Scenario': 'Both wrong', 'ATE': dr_both_wrong['estimate'], 'SE': dr_both_wrong['se'],
     'CI_lower': dr_both_wrong['ci_lower'], 'CI_upper': dr_both_wrong['ci_upper']},
])
display(dr_results.round(4))

In [ ]:
# Visualise the DR robustness
fig, ax = plt.subplots(figsize=(9, 5))

colors = ['#4DAD5B', '#FFD733', '#FFD733', '#EE1515']
y_pos = range(len(dr_results))

for i, (_, row) in enumerate(dr_results.iterrows()):
    ax.barh(i, row['ATE'], xerr=1.96*row['SE'], color=colors[i],
            edgecolor='white', height=0.6, capsize=5)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(dr_results['Scenario'])
ax.set_xlabel('Estimated ATE')
ax.set_title('Doubly Robust Property: Resilience to Misspecification')
ax.axvline(dr_correct['estimate'], color='gray', ls=':', lw=1.5, label='Both-correct benchmark')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
oak_says(
    "This is the <b>doubly robust property</b>: the AIPW estimator gives a consistent "
    "estimate as long as <em>at least one</em> of the two models (propensity score or outcome) "
    "is correctly specified. When both are wrong, all bets are off -- that is why "
    "careful model specification still matters, even with DR methods."
)

---
## 5.8 Sensitivity to Unobserved Confounding

In [ ]:
# Oster (2019)-style sensitivity: how would the coefficient change if R-squared
# increased toward a hypothetical R_max?
#
# The Oster bound: beta* = beta_long - delta * (beta_short - beta_long) * (R_max - R_long) / (R_long - R_short)
# where delta = 1 implies equal selection on observables and unobservables.

# Short model (naive, no controls)
short = smf.ols('badges ~ exp_share_used', data=trainers).fit()
# Long model (with confounders)
long = smf.ols('badges ~ exp_share_used + wealth + trainer_experience + strategy_score + play_hours',
               data=trainers).fit()

beta_short = short.params['exp_share_used']
beta_long = long.params['exp_share_used']
R_short = short.rsquared
R_long = long.rsquared

print(f"Short model: beta = {beta_short:.4f}, R2 = {R_short:.4f}")
print(f"Long model:  beta = {beta_long:.4f}, R2 = {R_long:.4f}")

# Sweep over R_max values
r_max_range = np.linspace(R_long + 0.01, min(1.0, R_long * 2.5), 100)
delta = 1.0  # proportional selection assumption

beta_star = beta_long - delta * (beta_short - beta_long) * (r_max_range - R_long) / (R_long - R_short + 1e-10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(r_max_range, beta_star, color='#EE1515', lw=2.5, label='Oster-adjusted coefficient')
ax.axhline(0, color='gray', ls=':', lw=1.5, label='Zero effect')
ax.axhline(beta_long, color='#3B4CCA', ls='--', lw=2, label=f'Controlled estimate = {beta_long:.4f}')
ax.axvline(R_long, color='#FFD733', ls=':', lw=1.5, label=f'Observed R2 = {R_long:.4f}')

# Find the R_max where beta_star = 0
# beta_long - delta * (beta_short - beta_long) * (R_zero - R_long) / (R_long - R_short) = 0
denom = delta * (beta_short - beta_long)
if abs(denom) > 1e-10:
    R_zero = R_long + beta_long * (R_long - R_short) / denom
    if R_long < R_zero < 1.0:
        ax.axvline(R_zero, color='#EE1515', ls='--', lw=1.5, alpha=0.7,
                   label=f'Effect = 0 at R_max = {R_zero:.4f}')
        ax.scatter([R_zero], [0], color='#EE1515', s=100, zorder=5)

ax.set_xlabel(r'Hypothetical $R^2_{max}$')
ax.set_ylabel('Adjusted coefficient on exp_share_used')
ax.set_title('Sensitivity Analysis: How Strong Must Unobserved Confounding Be?')
ax.legend(fontsize=9, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
oak_says(
    "This Oster-style sensitivity analysis asks: <em>how much residual variation "
    "would unobserved confounders need to explain before our estimated effect "
    "disappears?</em> If the answer is 'a lot more than what observables explain', "
    "our result is <b>robust</b> to moderate unobserved confounding. "
    "If the effect vanishes quickly, we should be worried."
)

---
## Challenges

Complete these exercises to earn the **Rainbow Badge**!

### Challenge 1: Manual OVB Calculation

The OVB formula states:

$$\text{OVB} = \hat{\gamma} \cdot \hat{\delta}$$

where $\hat{\gamma}$ is the coefficient of the omitted variable in the *long* regression,
and $\hat{\delta}$ is the coefficient from regressing the omitted variable on the treatment.

**Task:** For the omitted variable `wealth`, manually compute the OVB using the formula above
and verify it matches the actual bias (short coefficient - long coefficient).

In [ ]:
# Challenge 1: Manual OVB computation
# Step 1: Run the "long" regression: badges ~ dept_store_spending + wealth
long_reg = smf.ols('badges ~ dept_store_spending + wealth', data=trainers).fit()
gamma_hat = long_reg.params['wealth']  # coefficient on omitted var in long regression

# Step 2: Regress wealth on dept_store_spending (auxiliary regression)
aux_reg = smf.ols('wealth ~ dept_store_spending', data=trainers).fit()
delta_hat = aux_reg.params['dept_store_spending']  # relationship between treatment and omitted var

# Step 3: Compute OVB = gamma * delta
ovb_formula = gamma_hat * delta_hat

# Step 4: Verify against actual bias
short_reg = smf.ols('badges ~ dept_store_spending', data=trainers).fit()
actual_bias = short_reg.params['dept_store_spending'] - long_reg.params['dept_store_spending']

print(f"gamma (coef on wealth in long reg): {gamma_hat:.6f}")
print(f"delta (coef on dept_store in aux reg): {delta_hat:.6f}")
print(f"OVB from formula (gamma * delta): {ovb_formula:.6f}")
print(f"Actual bias (short - long coef):   {actual_bias:.6f}")
print(f"Match: {np.isclose(ovb_formula, actual_bias, atol=1e-8)}")

### Challenge 2: Stabilised IPW Weights

Standard IPW weights are $w_i = D_i / e(X_i) + (1 - D_i) / (1 - e(X_i))$.

**Stabilised** weights replace the numerator with the marginal treatment probability:
$w_i^{\text{stab}} = D_i \cdot P(D=1) / e(X_i) + (1 - D_i) \cdot P(D=0) / (1 - e(X_i))$

**Task:** Implement stabilised IPW weights and compare the ATE estimate to the standard
Hajek estimator.

In [ ]:
# Challenge 2: Stabilised IPW weights
p_d1 = treatment.mean()  # marginal probability of treatment
p_d0 = 1 - p_d1

# Stabilised weights
sw1 = treatment * p_d1 / ps
sw0 = (1 - treatment) * p_d0 / (1 - ps)

# Stabilised Hajek estimator
mu1_stab = np.sum(sw1 * outcome) / np.sum(sw1)
mu0_stab = np.sum(sw0 * outcome) / np.sum(sw0)
ate_stabilised = mu1_stab - mu0_stab

# Compare
print(f"Standard Hajek ATE:    {hajek_result['estimate']:.4f}")
print(f"Stabilised Hajek ATE:  {ate_stabilised:.4f}")
print(f"Difference:            {ate_stabilised - hajek_result['estimate']:.6f}")

# Compare weight distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
standard_w = np.where(treatment == 1, 1.0/ps, 1.0/(1-ps))
stabilised_w = np.where(treatment == 1, p_d1/ps, p_d0/(1-ps))

ax1.hist(standard_w, bins=40, color='#EE1515', alpha=0.7, edgecolor='white')
ax1.set_title(f'Standard Weights (var={standard_w.var():.2f})')
ax1.set_xlabel('Weight')

ax2.hist(stabilised_w, bins=40, color='#3B4CCA', alpha=0.7, edgecolor='white')
ax2.set_title(f'Stabilised Weights (var={stabilised_w.var():.2f})')
ax2.set_xlabel('Weight')

fig.suptitle('Standard vs Stabilised IPW Weights', fontweight='bold')
fig.tight_layout()
plt.show()

### Challenge 3: Doubly Robust with a Different Treatment-Outcome Pair

**Task:** Estimate the causal effect of `cave_training` (binary) on `total_battles_won`,
using confounders `trainer_experience`, `strategy_score`, `play_hours`, and `dedication`.
Use the `doubly_robust()` function.

In [ ]:
# Challenge 3: DR estimation for cave_training -> total_battles_won
treat_c3 = trainers['cave_training'].values
outcome_c3 = trainers['total_battles_won'].values
X_c3 = trainers[['trainer_experience', 'strategy_score', 'play_hours', 'dedication']].values

dr_c3 = doubly_robust(outcome_c3, treat_c3, X_c3)

print("=== Doubly Robust: cave_training -> total_battles_won ===")
print(f"  ATE: {dr_c3['estimate']:.4f}")
print(f"  SE:  {dr_c3['se']:.4f}")
print(f"  95% CI: [{dr_c3['ci_lower']:.4f}, {dr_c3['ci_upper']:.4f}]")

# Compare with naive difference in means
naive_c3 = trainers.loc[treat_c3==1, 'total_battles_won'].mean() - trainers.loc[treat_c3==0, 'total_battles_won'].mean()
print(f"\nNaive difference in means: {naive_c3:.4f}")
print(f"DR-adjusted ATE:           {dr_c3['estimate']:.4f}")
print(f"Confounding bias removed:  {naive_c3 - dr_c3['estimate']:.4f}")

### Challenge 4: Sensitivity Analysis -- At What Point Does the Effect Vanish?

**Task:** For the `exp_share_used -> badges` relationship (with the full set of controls),
compute the Oster delta: how proportionally important would unobservables need to be
(relative to observables) to explain away the effect entirely?

Use $R_{\max} = 1.3 \times R_{\text{long}}$ (Oster's recommended bound) and solve for $\delta^*$.

In [ ]:
# Challenge 4: Compute Oster's delta*
# beta* = beta_long - delta * (beta_short - beta_long) * (R_max - R_long) / (R_long - R_short)
# Set beta* = 0 and solve for delta:
# delta* = beta_long * (R_long - R_short) / ((beta_short - beta_long) * (R_max - R_long))

R_max_oster = 1.3 * R_long  # Oster's recommended bound

numerator = beta_long * (R_long - R_short)
denominator = (beta_short - beta_long) * (R_max_oster - R_long)

if abs(denominator) > 1e-10:
    delta_star = numerator / denominator
else:
    delta_star = float('inf')

print(f"Short model: beta = {beta_short:.4f}, R2 = {R_short:.4f}")
print(f"Long model:  beta = {beta_long:.4f}, R2 = {R_long:.4f}")
print(f"R_max (1.3 * R_long): {R_max_oster:.4f}")
print(f"")
print(f"Oster delta* = {delta_star:.4f}")
print()
if abs(delta_star) > 1:
    print("delta* > 1 means unobservables would need to be MORE important than")
    print("all observables combined to explain away the effect. The result is robust!")
else:
    print(f"delta* = {delta_star:.2f} < 1 means unobservables with just {abs(delta_star)*100:.0f}%")
    print("of the importance of observables could explain away the effect. Caution warranted.")

---
## Rainbow Badge Earned!

In [ ]:
badge_earned("Rainbow Badge", 5)